> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.14 - ORM

## Exercícios

#### Q1. Conhecendo os dados
Baixe o seguinte csv onde iremos trabalhar. Ele contém informações sobre salários de profissionais de dados de uma empresa hipotética entre 2009 e 2016
* https://github.com/camilalaranjeira/python-intermediario/blob/main/salaries.csv

Suas colunas, descritas na [página do Kaggle que contém o dataset](https://www.kaggle.com/datasets/krishujeniya/salary-prediction-of-data-professions?resource=download), são:
* FIRST NAME: Primeiro nome do profissional de dados (String)
* LAST NAME: Sobrenome do profissional de dados (String)
* SEX: Gênero do profissional de dados (String: 'F' para Feminino, 'M' para Masculino)
* DOJ (Date of Joining): A data em que o profissional de dados ingressou na empresa (Data no formato MM/DD/AAAA)
* CURRENT DATE: A data atual ou a data de referência dos dados (Data no formato MM/DD/AAAA)
* DESIGNATION: O cargo ou designação do profissional de dados (String: ex., Analista, Analista Sênior, Gerente)
* AGE: Idade do profissional de dados (Integer)
* SALARY: Salário anual do profissional de dados (Float)
* UNIT: Unidade de negócios ou departamento em que o profissional de dados trabalha (String: ex., TI, Finanças, Marketing)
* LEAVES USED: Número de licenças utilizadas pelo profissional de dados (Integer)
* LEAVES REMAINING: Número de licenças restantes para o profissional de dados (Integer)
* RATINGS: Avaliações de desempenho do profissional de dados (Float)
* PAST EXP: Experiência de trabalho anterior em anos antes de ingressar na empresa atual (Float)

Na célula a seguir, **carregue os dados do CSV e dê uma olhada neles antes de seguir**.

In [ ]:
import pandas as pd

# Carregando o arquivo CSV
df = pd.read_csv('salaries.csv')

# Convertendo as colunas de data para o formato datetime do Pandas
df['DOJ'] = pd.to_datetime(df['DOJ'])
df['CURRENT DATE'] = pd.to_datetime(df['CURRENT DATE'])

# Visualizando as primeiras linhas
display(df.head())

# (Opcional) Descobrindo os valores únicos para montar as classes Enum na próxima questão
print("Valores únicos em SEX:", df['SEX'].unique())
print("Valores únicos em DESIGNATION:", df['DESIGNATION'].unique())
print("Valores únicos em UNIT:", df['UNIT'].unique())

#### Q2. Modelando os dados
Você deve **criar um ORM com SQLAlchemy capaz de comportar os dados dessa base**.

* Crie um campo de chave primária `ID`, que deve ser incrementado automaticamente
* Os campos SEX, DESIGNATION e UNIT devem ser definidos como classes `Enum` com os possíveis valores (consulte os valores únicos dessas colunas)
* Para os outros campos, consulte os tipos de dados informados na descrição acima

In [ ]:
import enum
from sqlalchemy import Column, Integer, String, Float, Date, Enum
from sqlalchemy.orm import declarative_base

Base = declarative_base()

# 1. Definindo as classes Enum
class SexEnum(enum.Enum):
    F = 'F'
    M = 'M'

class DesignationEnum(enum.Enum):
    Analyst = 'Analyst'
    Senior_Analyst = 'Senior Analyst'
    Manager = 'Manager'
    Senior_Manager = 'Senior Manager'
    Director = 'Director'
    Vice_President = 'Vice President'

class UnitEnum(enum.Enum):
    IT = 'IT'
    Finance = 'Finance'
    Marketing = 'Marketing'
    Management = 'Management'
    Operations = 'Operations'
    Web = 'Web'

# 2. Criando o modelo ORM
class Profissional(Base):
    __tablename__ = 'profissionais'

    # Note que passamos o nome exato da coluna no banco como primeiro argumento
    id = Column('ID', Integer, primary_key=True, autoincrement=True)
    first_name = Column('FIRST NAME', String)
    last_name = Column('LAST NAME', String)
    sex = Column('SEX', Enum(SexEnum))
    doj = Column('DOJ', Date)
    current_date = Column('CURRENT DATE', Date)
    designation = Column('DESIGNATION', Enum(DesignationEnum))
    age = Column('AGE', Integer)
    salary = Column('SALARY', Float)
    unit = Column('UNIT', Enum(UnitEnum))
    leaves_used = Column('LEAVES USED', Integer)
    leaves_remaining = Column('LEAVES REMAINING', Integer)
    ratings = Column('RATINGS', Float)
    past_exp = Column('PAST EXP', Float)

#### Q3. Estabelecendo uma conexão

Usando o método `create_engine` do SQLAlchemy, crie uma conexão com um novo banco de dados SQLite chamado `salarios`.

In [ ]:
from sqlalchemy import create_engine

# Criando a conexão com o banco SQLite. O echo=False omite os logs extensos do SQL no terminal.
engine = create_engine('sqlite:///salarios.db', echo=False)

#### Q4. Criando as tabelas
Crie as tabelas da questão Q2 no banco `salarios`.

In [ ]:
# Cria a tabela 'profissionais' no arquivo salarios.db
Base.metadata.create_all(engine)
print("Tabelas criadas com sucesso!")

#### Q5. Populando

Usando o método `to_sql` da biblioteca Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html)), popule o banco `salarios` com os dados do csv que você carregou na questão Q1.
* Lembre-se de definir o parâmetro `if_exists='append'` para que as tabelas não sejam dropadas e recriadas.

In [ ]:
# index=False evita que o índice do DataFrame seja exportado como uma coluna extra
# if_exists='append' adiciona os dados à tabela existente que criamos no passo Q4
df.to_sql('profissionais', con=engine, if_exists='append', index=False)

print("Banco populado com sucesso!")

#### Q6. Consultas SQL vs ORM

Agrupe os dados por DESIGNATION e selecione o mínimo, máximo e a média dos salários (SALARY) divididos por 12. Já que o atributo SALARY é anual, dividir por 12 nos mostrará os valores mensais.

Assumindo que a variável que armazena a sua conexão se chama `engine`, você deve realizar a query acima de três formas:
* Executando a query SQL através de uma instância de conexão retornada pelo método `engine.connect()`
* Executando a query SQL com o método `read_sql_query` do Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)). Você usará mesma instância `engine.connect()` como um dos parâmetros.
* Executando uma query criada com o módulo `select` do SQLAlchemy. Sua execução deve ser feita através de um objeto `Session` do módulo `orm` do SQLAlchemy (`Session(engine)`).


In [ ]:
from sqlalchemy import text

query_sql = text("""
    SELECT DESIGNATION, 
           MIN(SALARY / 12.0) AS min_mensal, 
           MAX(SALARY / 12.0) AS max_mensal, 
           AVG(SALARY / 12.0) AS avg_mensal
    FROM profissionais
    GROUP BY DESIGNATION
""")

print("--- RESULTADOS VIA ENGINE.CONNECT (SQL) ---")
with engine.connect() as conn:
    resultado = conn.execute(query_sql)
    for linha in resultado:
        print(linha)

In [ ]:
print("--- RESULTADOS VIA PANDAS (READ_SQL_QUERY) ---")
# O Pandas cuida de abrir e fechar a conexão, transformando o resultado num DataFrame bonito
with engine.connect() as conn:
    df_resultado = pd.read_sql_query(query_sql, conn)

display(df_resultado)

In [ ]:
from sqlalchemy.orm import Session
from sqlalchemy import select, func

print("--- RESULTADOS VIA SESSÃO ORM ---")

with Session(engine) as session:
    stmt = select(
        Profissional.designation,
        func.min(Profissional.salary / 12.0).label('min_mensal'),
        func.max(Profissional.salary / 12.0).label('max_mensal'),
        func.avg(Profissional.salary / 12.0).label('avg_mensal')
    ).group_by(Profissional.designation)
    
    resultados_orm = session.execute(stmt).all()
    
    for row in resultados_orm:
        # row[0] contém o objeto Enum, por isso pegamos row[0].value para ler a string legível
        designation = row[0].value if hasattr(row[0], 'value') else row[0]
        print(f"Cargo: {designation} | Min: {row[1]:.2f} | Max: {row[2]:.2f} | Avg: {row[3]:.2f}")